In [1]:
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
load_dotenv(override=True)
MODEL='gpt-4o-mini'
openai = OpenAI()

In [ ]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
  """
  A utility class to represent a Website that we have scraped, now with links
  """

  def __init__(self, url):
    self.url = url
    response = requests.get(url, headers=headers)
    self.body = response.content
    soup = BeautifulSoup(self.body, 'html.parser')
    self.title = soup.title.string if soup.title else "No title found"

    if soup.body:
      for irrelevant in soup.body(['script', 'style', 'img', 'input']):
        irrelevant.decompose()
      self.text = soup.body.get_text(separator="\n", strip=True)
    else:
      self.text = ""

    links = [link.get('href') for link in soup.find_all('a')]
    self.links = [link for link in links if link]

  def get_contents(self):
    return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [4]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/',
 'https://edwarddonner.com/2025/04/21/the-

In [5]:
# Deciding which links relevant
link_system_prompt = "You are provided with a list of links found on a webpage. \
  You are able to decide which of the links would be most relevant to include in a brochure about the company, \
  such as links to an About page, or Company page, or Careers/Job pages.\n"

link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
  "links" : [
    { "type", "about page", "url": "https://full.url/goes/here/about" },
    { "type", "careers page", "url": "https://another.full.url/careers" }
  ]
}
"""

In [6]:
def get_links_user_prompt(website):
  user_prompt = f"Here is the list of links on the website of {website.url} - "
  user_prompt += "please decide which of these are relevant wen links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of service, Privacy, email links.\n"
  user_prompt += "Links (some might be relative links):\n"
  user_prompt += "\n".join(website.links)
  return user_prompt

In [21]:
def get_links(url):
  website = Website(url)
  response = openai.chat.completions.create(
    model=MODEL,
    messages=[
      { "role": "system", "content": link_system_prompt },
      { "role": "user", "content": get_links_user_prompt(website) }
    ],
    response_format = { "type" : "json_object" }
  )
  result = response.choices[0].message.content
  return json.loads(result)

In [22]:
huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/deepseek-ai/DeepSeek-R1-0528',
 '/ResembleAI/chatterbox',
 '/google/gemma-3n-E4B-it-litert-preview',
 '/deepseek-ai/DeepSeek-R1-0528-Qwen3-8B',
 '/Qwen/Qwen3-Embedding-0.6B-GGUF',
 '/models',
 '/spaces/ResembleAI/Chatterbox',
 '/spaces/enzostvs/deepsite',
 '/spaces/multimodalart/wan2-1-fast',
 '/spaces/wushuang98/Direct3D-S2-v1.0-demo',
 '/spaces/alexnasa/Chain-of-Zoom',
 '/spaces',
 '/datasets/yandex/yambda',
 '/datasets/open-r1/Mixture-of-Thoughts',
 '/datasets/fka/awesome-chatgpt-prompts',
 '/datasets/open-thoughts/OpenThoughts3-1.2M',
 '/datasets/Hcompany/WebClick',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel',
 '/microsoft',
 '/grammarly',
 '/Write

In [23]:
get_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'community discussion', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}

In [24]:
# Make the brochure
def get_all_details(url):
  result = "Landing page:\n"
  result += Website(url).get_contents()
  links = get_links(url)
  print("Found links: ", links)
  for link in links['links']:
    result += f"\n\n{link['type']}\n"
    result += Website(link['url']).get_contents()
  return result

In [25]:
print(get_all_details("https://huggingface.co"))

Found links:  {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'company page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'documentation page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}
Landing page:
Webpage Title:
Hugging Face – The AI community building the future.
Webpage Contents:
Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community co

In [26]:
system_prompt = "You are an assistant that analyses the contents of several relevant pages from a company website \
  and creates a short brochure about the company for prospective customers, investors and recruits. Respond in \
  Markdown. Include details of company culture, customers and careers/jobs if you hve the information."

In [27]:
def get_brochure_user_prompt(company_name, url):
  user_prompt = f"You are looking at a company called: {company_name}\n"
  user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
  user_prompt += get_all_details(url)
  user_prompt = user_prompt[:5_000] # only 5000 chars
  return user_prompt

In [28]:
get_brochure_user_prompt("Huggingface", "https://huggingface.co")

Found links:  {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'learn page', 'url': 'https://huggingface.co/learn'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community discussion', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}


'You are looking at a company called: Huggingface\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-R1-0528\nUpdated\n8 days ago\n•\n74.7k\n•\n1.8k\nResembleAI/chatterbox\nUpdated\n8 days ago\n•\n649\ngoogle/gemma-3n-E4B-it-litert-preview\nUpdated\n11 days ago\n•\n956\ndeepseek-ai/DeepSeek-R1-0528-Qwen3-8B\nUpdated\n8 days ago\n•\n139k\n•\n686\nQwen/Qwen3-Embedding-0.6B-GGUF\nUpdated\nabout 15 hours ago\n•\n2.43k\n•\n193\nBrowse 1M+ models\nSpaces\nRunning\non\nZero\

In [29]:
def create_brochure(company_name, url):
  response = openai.chat.completions.create(
    model=MODEL,
    messages=[
      { "role": "system", "content": system_prompt },
      { "role": "user", "content": get_brochure_user_prompt(company_name, url) }
    ]
  )
  result = response.choices[0].message.content
  display(Markdown(result))

In [30]:
create_brochure("Huggingface", "https://huggingface.co")

Found links:  {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'company page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


# Hugging Face Brochure

## Welcome to Hugging Face
**"The AI community building the future."**

At Hugging Face, we are a dedicated community focused on empowering the machine learning landscape. Our platform serves as a collaborative space for creators, developers, researchers, and businesses to access, share, and develop state-of-the-art machine learning models and datasets.

---

## Our Offerings

### Models 
Explore over **1 million pre-trained models** and discover trending architectures inclusive of diverse applications.

### Datasets 
Browse an extensive collection of **250,000+ datasets** tailored for various machine learning tasks.

### Spaces 
Run and collaborate on applications in a streamlined way with our interactive **Spaces**. 

---

## Who We Serve
Hugging Face supports over **50,000 organizations**, including some of the leading names in tech like:

- **Google**
- **Amazon**
- **Microsoft**
- **Meta**

Our community is built on partnerships and collations, ensuring that every member benefits a wealth of resources and collaboration opportunities.

---

## Our Culture
At Hugging Face, we embrace a culture of **openness and collaboration**. We're on a mission to democratize machine learning, ensuring that anyone can harness its potential. Our contributors play a vital role in our vibrant community, continually pushing the boundaries of what's possible with machine learning.

We also emphasize:

- **Innovation:** Our dedicated open-source projects ensure that we're consistently on the cutting edge of technology.
- **Diversity:** We welcome individuals from various backgrounds to uniquely contribute in their way.

---

## Career Opportunities
Join a dynamic team passionate about shaping the future of AI. At Hugging Face, we’re looking for talented individuals who are:

- Innovative thinkers
- Team players
- Passionate about machine learning and its applications

Explore our latest job openings and discover how you can be part of our journey at [Hugging Face Careers](https://huggingface.co/jobs).

---

## Join the Revolution
Whether you are a potential customer seeking powerful AI solutions, an investor looking to support innovative technology, or a future recruit aspiring to work in a collaborative environment, **Hugging Face** welcomes you to join us on this groundbreaking journey in AI!

For more information, sign up or log in at [Hugging Face](https://huggingface.co).

---

> For queries, you can reach out to us via our social platforms: [Twitter](https://twitter.com/huggingface), [LinkedIn](https://linkedin.com/company/huggingface), or [Discord](https://discord.com/invite/huggingface). 

Let’s build the future of AI together!

In [ ]:
# Streaming way
def stream_brochure(company_name, url):
  stream = openai.chat.completions.create(
    model=MODEL,
    messages=[
      { "role": "system", "content": system_prompt },
      { "role": "user", "content": get_brochure_user_prompt(company_name, url) }
    ],
    stream=True
  )

  response = ""
  display_handle = display(Markdown(""), display_id=True)
  for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```", "").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

In [32]:
stream_brochure("Huggingface", "https://huggingface.co")

Found links:  {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'join page', 'url': 'https://huggingface.co/join'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


# Hugging Face Company Brochure

## Welcome to Hugging Face
**The AI Community Building the Future.**  
At Hugging Face, we are at the forefront of machine learning, offering a collaborative platform where developers, researchers, and enthusiasts come together to create, share, and learn about AI technologies. Our mission is to democratize machine learning and make it accessible to everyone.

---

## What We Offer
### **1. Machine Learning Models and Datasets**
- Explore over **1M+ models** across various applications in natural language processing, image recognition, audio processing, and more.
- Access **250k+ datasets** that facilitate focused research and innovation in the AI field.

### **2. Spaces for Application Development**
- Create and run applications that leverage state-of-the-art models in our **Spaces** platform, allowing users to demonstrate their AI solutions effectively.
- **Examples of spaces include**:
  - **Chatterbox TTS**: Expressive text-to-speech application
  - **DeepSite**: Generate applications using deep learning
  - **Chain-of-Zoom**: Extreme super-resolution technology

### **3. Enterprise Solutions**
- We provide enterprise-grade security and dedicated support with tailored solutions to meet the unique needs of businesses.
- Companies like **Google**, **Microsoft**, and **Amazon** trust Hugging Face for their machine learning requirements.

---

## Customer Base
More than **50,000 organizations** partner with us, including:

- **Meta**
- **Intel**
- **Grammarly**
- **Writer**

Our community comprises everyone from individual researchers to large enterprises, all collaborating on the next generation of machine learning technologies.

---

## Company Culture
At Hugging Face, we believe in an inclusive and supportive work environment that fosters innovation, collaboration, and continuous learning:

- **Community-Driven**: We thrive on collaboration and constantly engage with our community to improve our tools and technologies.
- **Open Source**: We develop key tools such as Transformers, Diffusers, and more to contribute to the broader AI community.
- **Continuous Learning**: With a dedicated team of over **215 skilled professionals**, we encourage personal and professional growth for each employee.

---

## Careers at Hugging Face
Join us in shaping the future of AI! We are always looking for talented individuals across various disciplines, including:

- Software Development
- Data Science and Machine Learning
- Product Management
- Community Engagement

Explore current job openings on our [Jobs Page](https://huggingface.co/jobs) and become a part of our journey to democratize AI.

---

## Connect with Us
Stay up-to-date with the latest developments, updates, and community events:

- [Website](https://huggingface.co)
- [Twitter](https://twitter.com/huggingface)
- [LinkedIn](https://linkedin.com/company/huggingface)
- [Discord](https://discord.gg/huggingface)

**Hugging Face** – Delivering advanced AI solutions with a human touch. Join us in creating the future of technology through collaboration and innovation!